# Kaggle submission – Hull Tactical Market Prediction

This notebook runs **on Kaggle** (not Colab). It loads the reusable code from `src/` of this repository,
fits the selected baseline on the full training data and answers the competition's evaluation API one day at a time.

**Setup on Kaggle (once):**
1. Upload the four files in `src/` (`__init__.py`, `data.py`, `models.py`, `utils.py`) as a Kaggle Dataset named `group60-src`.
2. Open the competition → *Code* → *New Notebook* → *File → Import Notebook* and upload this file.
3. In the right panel: *Add Input* → add the `group60-src` dataset (the competition data is already attached).
4. Turn **Internet off** (required for code competitions), *Save Version*, then *Submit* from the notebook viewer.

In [ ]:
import glob, os, shutil, sys

INPUT_ROOT = os.environ.get('KAGGLE_INPUT_ROOT', '/kaggle/input')
WORK_DIR = os.environ.get('KAGGLE_WORK_DIR', '/kaggle/working')

# Competition data and the evaluation API
COMP_DIR = os.path.dirname(sorted(glob.glob(f'{INPUT_ROOT}/**/hull-tactical-market-prediction/train.csv', recursive=True))[0])
sys.path += [COMP_DIR, os.path.join(COMP_DIR, 'kaggle_evaluation')]

# Our src/ package, uploaded as a Kaggle Dataset -> copy into a proper package folder
src_files = glob.glob(f'{INPUT_ROOT}/**/models.py', recursive=True)
assert src_files, 'Add the group60-src dataset as an input'
SRC_IN = os.path.dirname(src_files[0])
os.makedirs(f'{WORK_DIR}/src', exist_ok=True)
for f in ['__init__.py', 'data.py', 'models.py', 'utils.py']:
    if os.path.exists(os.path.join(SRC_IN, f)):
        shutil.copy(os.path.join(SRC_IN, f), f'{WORK_DIR}/src/{f}')
open(f'{WORK_DIR}/src/__init__.py', 'a').close()
sys.path.insert(0, WORK_DIR)
print('competition data:', COMP_DIR)
print('src copied from  :', SRC_IN)

In [ ]:
import numpy as np
import pandas as pd
import polars as pl

from src import data as D
from src import models as M

# Best baseline in walk-forward validation (see assignment1_eda_baseline.ipynb, section 6)
MODEL_NAME = 'vol_target_21d'

train = D.add_lagged_returns(D.drop_early_rows(D.load_train(COMP_DIR)))
model = M.baseline_zoo()[MODEL_NAME].fit(train)
predictor = M.OnlinePredictor(model, train)
print('model:', MODEL_NAME, '| trained on', len(train), 'days')

In [ ]:
def predict(test: pl.DataFrame) -> float:
    """Called by the evaluation API once per trading day; returns the position in [0, 2]."""
    return predictor.predict_one(test.to_pandas())

In [ ]:
import kaggle_evaluation.default_inference_server

inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway((COMP_DIR,))

In [ ]:
# Local dry run writes the predictions to submission.parquet
if os.path.exists('submission.parquet'):
    print(pd.read_parquet('submission.parquet'))